# Zynq-7020 / B35 inspection and validation

用于在板端 Jupyter 检查新工程、验证真实寄存器和 UDP。默认只读；运行全部单元格不会加载位流或启动服务。与本 Notebook 配套的 b35_pc_update.zip 在离线回归完成后生成。

此流程不连接机械臂，不调用 CAN、不使能或发送动作。合成阶段/视觉验证只证明 PL 路径，不能视作真实毛巾任务成功或现场标定。原 B32 工程作为回退基线保留。


设备树PS参考频率为33.333333MHz，但PYNQ 2.7默认按50MHz计算。先调用board.clock_reference.configure_reference校准软件计算；原始bit/HWH不改，分频仍为5×4。必须核验物理FCLK约50MHz及PL的1.5秒超时。


In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, zipfile, subprocess
PROGRAM_B35 = False
RUN_SYNTHETIC_PL_TEST = False
START_UDP_SERVER = False
TEST_LCD = False
PACKAGE = Path.cwd() / 'b35_pc_update.zip'
INSTALL = Path('/home/xilinx/jupyter_notebooks/fpga_fpg1_v3_b35_20261113')
PC_IP = '192.168.2.10'
print({'uid': os.getuid(), 'python': sys.version, 'cwd': str(Path.cwd()),
       'program_b35': PROGRAM_B35, 'start_udp_server': START_UDP_SERVER})


## 1. Inspect the board
只读取系统、网络、PYNQ 和当前 Overlay 元数据，不访问未知 AXI 地址。

In [ ]:
import pynq
from pynq import PL, Overlay
print('PYNQ:', pynq.__version__)
print('Current bitfile:', PL.bitfile_name)
print('Known IP:', {k: {'phys_addr': v.get('phys_addr'), 'addr_range': v.get('addr_range')} for k,v in PL.ip_dict.items()})
print(subprocess.check_output(['ip', '-br', 'address'], text=True))
print(subprocess.check_output(['ss', '-lunp'], text=True))


## 2. Verify and unpack the release
上传配套 ZIP 后执行。本步骤只写一个新的版本目录，拒绝覆盖旧目录；所有文件对照包内 manifest 核验。

In [ ]:
assert PACKAGE.is_file(), 'Upload b35_pc_update.zip beside this Notebook first'
with zipfile.ZipFile(PACKAGE) as archive:
    names = archive.namelist()
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names)
    assert sum(i.file_size for i in archive.infolist()) < 100_000_000
    manifest = json.loads(archive.read('release_manifest.json'))
    for row in manifest['files']:
        data = archive.read(row['path'])
        assert len(data) == row['bytes'] and hashlib.sha256(data).hexdigest() == row['sha256'], row['path']
    if INSTALL.exists():
        for row in manifest['files']:
            assert (INSTALL / row['path']).is_file()
            assert hashlib.sha256((INSTALL / row['path']).read_bytes()).hexdigest() == row['sha256'], row['path']
        print('Existing installed version matches; no files overwritten')
    else:
        INSTALL.mkdir()
        archive.extractall(INSTALL)
print('Verified files:', len(manifest['files']))
print('Release source:', manifest.get('qualification'))
sys.path.insert(0, str(INSTALL))


## 3. Parse the new HWH without programming
确认新包能由当前 PYNQ 解析，并检查 guard、LCD 与时钟。

In [ ]:
BIT = INSTALL / 'overlay/lcd_framebuffer_fpg1_v3_b35_20261113_revision/fpg1_v3_b35_20261113_revision.bit'
assert hashlib.sha256(BIT.read_bytes()).hexdigest() == '4b8cafe8e742b78ba1448eb1ab29e7f781cd3edc18c8cc8d1b6c5f654e0d8d12'
from board.clock_reference import configure_reference
print(configure_reference())
parsed = Overlay(str(BIT), download=False)
print('IP:', parsed.ip_dict)
print('Clocks:', parsed.clock_dict)
assert 'guard' in parsed.ip_dict and parsed.ip_dict['guard']['phys_addr'] == 0x43C00000
PREVIOUS_BIT = PL.bitfile_name
print('Rollback candidate:', PREVIOUS_BIT)


## 4. Program B35 and read its real identity
只有把 PROGRAM_B35 改成 True 才会执行。需确认没有其他服务占用 PL；失败时尝试回退到上一有效位流或包内 B34。

In [ ]:
if not PROGRAM_B35:
    print('SKIP: PROGRAM_B35=False; current PL unchanged')
else:
    from board.ownership import exclusive_owner
    previous = Path(PREVIOUS_BIT) if PREVIOUS_BIT else None
    rollback = previous if previous and previous.is_file() and previous.resolve() != BIT.resolve() else INSTALL / 'rollback/fpg1_v3_b34_20261113_final.bit'
    with exclusive_owner():
        try:
            overlay = Overlay(str(BIT))
            from board.clock_reference import verify_fclk0
            print("Physical FCLK0 MHz:", verify_fclk0())
            build_id = overlay.guard.read(0)
            assert build_id == 0x46504735, hex(build_id)
            from host.fold_config import load, identity
            expected = identity(load())
            assert overlay.guard.read(0x98) == expected['crc32'], 'PHASE_CONFIG_CRC mismatch'
            print({'build_id': hex(build_id), 'watchdog_control': overlay.guard.read(0x4C),
                   'phase_state': overlay.guard.read(0x90), 'verify_state': overlay.guard.read(0xE0)})
            assert overlay.guard.read(0x4C) == 0, 'Diagnostic service must not feed/arm FWD1'
        except Exception:
            Overlay(str(rollback))
            raise


## 5. Run SYNTHETIC PL validation
只向 PL 输入合成向量，检查探针、OPEN/GRIP/FOLD、撤臂事件、至少三帧视觉稳定、查询恢复和新会话清状态。完全不触碰机械臂。

In [ ]:
if not RUN_SYNTHETIC_PL_TEST:
    print('SKIP: RUN_SYNTHETIC_PL_TEST=False')
else:
    assert PROGRAM_B35, 'Program and verify B35 first'
    from validation.validate_board import run
    from board.ownership import exclusive_owner
    with exclusive_owner():
        result = run(overlay.guard, INSTALL / 'validation_runs')
    print(json.dumps(result, indent=2))
    from validation.validate_revision import run as validate_revision
    with exclusive_owner():
        print(json.dumps(validate_revision(overlay.guard, INSTALL / 'validation_runs'), indent=2))


## 6. LCD interface check
只在 TEST_LCD=True 时启用屏幕并测试帧缓冲；寄存器/帧计数通过不能代替人工确认物理画面。

In [ ]:
if not TEST_LCD:
    print('SKIP: TEST_LCD=False')
else:
    assert PROGRAM_B35
    from validation.validate_board import check_lcd
    from board.ownership import exclusive_owner
    with exclusive_owner():
        print(json.dumps(check_lcd(overlay.lcd, INSTALL / 'validation_runs'), indent=2))


## 7. Start the version-matched UDP service
先检查 9009 端口和旧服务，随后启动独立版本目录内的服务。此服务只诊断，不发送机器人动作。当前固化视觉门限尚未现场标定。

In [ ]:
if not START_UDP_SERVER:
    print('SKIP: START_UDP_SERVER=False')
else:
    assert PROGRAM_B35
    from board.service_control import control
    print(json.dumps(control('start', peer=PC_IP, source='host_observation', lcd=TEST_LCD), indent=2))


## 8. Stop only the service started here
按需执行；不终止其他任务，不自动重启旧工程。

In [ ]:
# from board.service_control import control
# print(control('stop'))  # Stops only this release's registered service.


## Review notes
- B35修复断流过期、相邻帧间隔和视觉快照布局，已补齐配置CRC。
- 配置JSON、RTL常量及PS身份校验必须成套发布；修改任何阈值后需重新生成位流。
- 当前calibration_status=synthetic_test，只资格化合成特征；HOST模式禁用有效视觉证据。
- 本轮不发送机械臂动作，现场标定和闭环测试另行完成。
